In [1]:
# notebooks/04_generation_test.ipynb

In [13]:
%run notebook_init.py

In [14]:
from rag_pipeline.retriever.retrieve import query_chunks
from rag_pipeline.llm.generator import generate_answer, format_prompt, generate_answer_with_gate

In [16]:
query = "Where is Tesla headquarters located?"
chunks = query_chunks(query, top_k=2)

✅ Retrieved 2 relevant chunks (requested 2)


In [17]:
from rag_pipeline.retriever.retrieve import query_chunks

# Retrieve chunks related to the known query
q = "PriceWaterhouseCooper"
chunks = query_chunks(q, top_k=5)

# Print the section titles and first few lines of retrieved content
for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Section: {c['metadata']['section']}")
    print(c['text'][:500])

✅ Retrieved 5 relevant chunks (requested 5)

--- Chunk 1 ---
Section: ITEM 2. PROPERTIES
ITEM 2. PROPERTIES
We are headquartered in Austin, Texas. Our principal facilities include a large number of properties in North America, Europe and Asia utilized for
manufacturing and assembly, warehousing, engineering, retail and service locations, Supercharger sites and administrative and sales offices. Our facilities
are used to support both of our reporting segments, and are suitable and adequate for the conduct of our business. We generally lease such facilities with
the primary exception 

--- Chunk 2 ---
Section: ITEM 16. SUMMARY
2, LLC
Delaware
Three Rivers Solar Manager 3, LLC
Delaware
TM International C.V.
Netherlands
TM Sweden AB
Sweden
USB SolarCity Manager IV, LLC
Delaware
USB SolarCity Master Tenant IV, LLC
California
USB SolarCity Owner IV, LLC
California
Visigoth Solar 1, LLC
Delaware
Visigoth Solar Holdings, LLC
Delaware
Visigoth Solar Managing Member 1, LLC
Delaware
VPP Project 1

In [18]:
dryrun = True # Set to TRUE by default

In [ ]:
from rag_pipeline.verifier.hallucination_guard import sentence_overlap

def ask_and_verify():
    while True:
        q = input("Ask a question (or 'q' to quit): ")
        if q.lower() == 'q':
            break
        chunks = query_chunks(q)

        print("\n🔎 Retrieved Chunks (Debug Info):")
        for i, c in enumerate(chunks):
            metadata = c.get('metadata', {})
            print(f"\n  Chunk {i+1}:")
            print(f"    doc_id: {metadata.get('doc_id', 'unknown')}")
            print(f"    chunk_id: {metadata.get('chunk_id', 'unknown')}")
            print(f"    section: {metadata.get('section', 'unknown')}")
            print(f"    distance: {c.get('distance', 'N/A')}")
            print(f"    text preview: {c['text'][:200]}...")

        answer, abstained = generate_answer_with_gate(q, chunks, model_name="gpt-4o")
        print("\nAnswer:\n", answer)
        if abstained:
            print("(System abstained from answering)")

In [20]:
ask_and_verify()

✅ Retrieved 5 relevant chunks (requested 5)

🔎 Retrieved Chunks:

Chunk 0 (Section: ITEM 16. SUMMARY)
 Motors FL, Inc.
Florida
Tesla Motors HK Limited
Hong Kong
Tesla Motors Iceland ehf.
Iceland
Tesla Motors Ireland Limited
Ireland
Tesla Motors Israel Ltd.
Israel
Tesla Motors Japan GK
Japan
Tesla Motors Limited
United Kingdom
Tesla Motors Luxembourg S.à r.l.
Luxembourg
Tesla Motors MA, Inc.
Massachusetts
Tesla Motors Netherlands B.V.
Netherlands
Tesla Motors New York LLC
New York
Tesla Motors NL LLC
Delaware
Tesla Motors NV, Inc.
Nevada
Tesla Motors PA, Inc.
Pennsylvania
Tesla Motors Romania S.R...

Chunk 1 (Section: ITEM 1. BUSINESS)
) that contains
reports, proxy and information statements, and other information regarding issuers that file electronically. Our website is located at www.tesla.com, and
our reports, amendments thereto, proxy statements and other information are also made available, free of charge, on our investor relations website at
ir.tesla.com as soon as reasonably pr

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag_pipeline:Query: Where is Tesla HQ located?
INFO:rag_pipeline:  [1] unknown::ITEM 16. SUMMARY::chunk-unknown
INFO:rag_pipeline:  [2] unknown::ITEM 1. BUSINESS::chunk-unknown
INFO:rag_pipeline:  [3] unknown::ITEM 16. SUMMARY::chunk-unknown
INFO:rag_pipeline:  [4] unknown::ITEM 1. BUSINESS::chunk-unknown
INFO:rag_pipeline:  [5] unknown::ITEM 1. BUSINESS::chunk-unknown
INFO:rag_pipeline:Answer length: 56 chars



Answer:
 The provided document does not contain that information.


In [ ]:
# Use query_chunks() as single source of truth for retrieval
# This ensures consistent metadata and avoids mismatched Chroma instances

from rag_pipeline.retriever.retrieve import query_chunks

chunks = query_chunks("PricewaterhouseCoopers", top_k=5)
for i, c in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"doc_id: {c['metadata'].get('doc_id', 'unknown')}")
    print(f"chunk_id: {c['metadata'].get('chunk_id', 'unknown')}")
    print(f"section: {c['metadata'].get('section', 'unknown')}")
    print(f"distance: {c.get('distance', 'N/A')}")
    print(f"text preview: {c['text'][:300]}...")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
